# Fine-tuning Qwen pakai LoRA — Ekstraksi Transaksi Sembako

**Jalankan di Google Colab dengan GPU:**
1. Upload notebook ini ke [colab.research.google.com](https://colab.research.google.com)
2. Menu **Runtime → Change runtime type → T4 GPU** → Save
3. Di panel **Files** (ikon folder kiri), upload 2 file: `train.jsonl` dan `val.jsonl`
4. **Runtime → Run all** (total ±10–15 menit)

Hasil akhirnya: file `qwen-ekstraksi-lora.zip` (adapter LoRA) otomatis kedownload.

In [1]:
%pip install -q -U transformers peft trl datasets accelerate

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
assert os.path.exists("train.jsonl") and os.path.exists("val.jsonl"), \
    "Upload dulu train.jsonl & val.jsonl lewat panel Files di kiri!"

from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})
print(ds)
print("\nContoh:", ds["train"][0]["messages"][1]["content"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2500
    })
    val: Dataset({
        features: ['messages'],
        num_rows: 250
    })
})

Contoh: tadi laku air mineral le minerale 600ml lima botol 14.000, lilin (pack) satu kg 12.000


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 0.5B: paling cepat buat training di T4 (cocok buat demo/tugas)
# ganti ke "Qwen/Qwen2.5-1.5B-Instruct" kalau mau lebih akurat (training lebih lama)
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype="auto", device_map="auto"
)
print("Model loaded:", MODEL_NAME)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct


In [4]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# LoRA = nge-train "tempelan" kecil (adapter), bukan seluruh model.
# r=16 itu ukuran adapternya; makin besar makin ekspresif tapi makin lambat.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir="hasil-training",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    logging_steps=20,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds["train"],
    peft_config=lora_config,
    args=args,
    processing_class=tokenizer,
)
trainer.train()
# Catatan: kalau ada error soal argumen (versi trl beda), jalankan ulang
# cell install paling atas dengan: %pip install -q -U trl  lalu Restart runtime

Tokenizing train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,1.494750
40,0.544831
60,0.445648
80,0.381916
100,0.350612
120,0.355560
140,0.322857
160,0.313172
180,0.298658
200,0.279034


KeyboardInterrupt: 

In [ ]:
# Evaluasi cepat: bandingkan output model vs label di 30 contoh validasi
import json, torch

model_ft = trainer.model
model_ft.eval()

SYSTEM_PROMPT = ds["val"][0]["messages"][0]["content"]

def prediksi(kalimat: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": kalimat},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model_ft.device)
    with torch.no_grad():
        out = model_ft.generate(**inputs, max_new_tokens=256, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

benar = 0
N = 30
for i in range(N):
    contoh = ds["val"][i]
    kalimat = contoh["messages"][1]["content"]
    label = json.loads(contoh["messages"][2]["content"])
    try:
        hasil = json.loads(prediksi(kalimat).replace("```json", "").replace("```", "").strip())
        if hasil == label:
            benar += 1
    except Exception:
        pass
    if i < 3:
        print("Kalimat :", kalimat)
        print("Label   :", label)
        print("Model   :", prediksi(kalimat))
        print()

print(f"Akurasi exact-match: {benar}/{N} = {benar/N*100:.0f}%")

In [ ]:
# Simpan adapter LoRA & download
trainer.save_model("qwen-ekstraksi-lora")
tokenizer.save_pretrained("qwen-ekstraksi-lora")

!zip -rq qwen-ekstraksi-lora.zip qwen-ekstraksi-lora

from google.colab import files
files.download("qwen-ekstraksi-lora.zip")

## Cara pakai adapter hasil training di `qwen_extractor.py`

Extract `qwen-ekstraksi-lora.zip`, taruh foldernya di sebelah `qwen_extractor.py`,
lalu di fungsi `_load()` ubah bagian load model jadi:

```python
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto")
_model = PeftModel.from_pretrained(base, "qwen-ekstraksi-lora")
```

(dan samakan `MODEL_NAME` dengan model dasar yang dipakai saat training di notebook ini)